# P3: Acquisition & Integration

**DSA 405**

**Lindsey Ramsey**

## Project Overview

My project analyzes how average hourly wages across eight U.S. industries changed relative to inflation between 2015 and 2025. The overall goal is to find whether wage growth consistently rose with inflation, as well as detecting if these patterns differed across industries.

My Two datasets:

1. **Bureau of Labor Statistics (BLS):** Monthly average hourly earnings for eight U.S. industries, which I audited and cleaned during P2.
2. **Federal Reserve Economic Data (FRED):** Monthly Consumer Price Index (CPIAUCSL) data, which measures changes in consumer prices over time.


In [25]:

import pandas as pd
import numpy as np
import requests
import time
from pathlib import Path
from io import BytesIO
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; educational project)"}
print("Successful")
print("Ready.")


Successful
Ready.


## 1. Data Acquisition

### 1.1 CPI Data from FRED

The CPI dataset is retrieved from the FRED CSV. The acquisition function checks for a cached file before making a new request, also uses a timeout, and validates the response.

The dataset contains 132 monthly observations from January 2015 to December 2025. The raw CSV is saved in data/raw/.

In [27]:
import pandas as pd
from io import BytesIO
import requests
import time

FRED_URL = ("https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=CPIAUCSL&cosd=2015-01-01&coed=2025-12-01")

CPI_RAW_PATH = RAW_DIR / "CPIAUCSL.csv"
def validate_cpi(content):
    try:
        df = pd.read_csv(BytesIO(content))
    except Exception as e:
        raise ValueError("CPI File Error.") from e
    if df.empty:
        raise ValueError("CPI dataset = empty.")
    if "CPIAUCSL" not in df.columns:
        raise ValueError("A column = missing.")
    if not ({"DATE", "observation_date"} & set(df.columns)):
        raise ValueError("CPI date column = missing.")
    return df
def acquire_cpi():
    if CPI_RAW_PATH.exists():
        print("Using cached CPI data:", CPI_RAW_PATH)
        content = CPI_RAW_PATH.read_bytes()
        return validate_cpi(content)
    print("Downloading CPI data from FRED")
    try:
        response = requests.get(FRED_URL, headers=HEADERS, timeout=30)
        response.raise_for_status()
        cpi = validate_cpi(response.content)
        CPI_RAW_PATH.write_bytes(response.content)
        print("Download successful!")
        print("Saved to:", CPI_RAW_PATH)
        return cpi
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"FRED data failed: {e}") from e
    finally:
        time.sleep(1)
cpi_raw = acquire_cpi()
print("CPI dataset shape:", cpi_raw.shape)
print("Columns:", cpi_raw.columns.tolist())
display(cpi_raw.head())


Using cached CPI data: data/raw/CPIAUCSL.csv
CPI dataset shape: (132, 2)
Columns: ['observation_date', 'CPIAUCSL']


,observation_date,CPIAUCSL
0,2015-01-01,234.747
1,2015-02-01,235.342
2,2015-03-01,235.976
3,2015-04-01,236.222
4,2015-05-01,237.001


In [28]:

cpi_check = cpi_raw.copy()
if "DATE" in cpi_check.columns:
    cpi_check = cpi_check.rename(columns={"DATE": "observation_date"})

cpi_check["date"] = pd.to_datetime(cpi_check["observation_date"], errors="raise")
cpi_check["CPIAUCSL"] = pd.to_numeric(cpi_check["CPIAUCSL"], errors="raise")
print("Total observations:", len(cpi_check))
print("Earliest date:", cpi_check["date"].min())
print("Latest date:", cpi_check["date"].max())
print("Missing values:")
print(cpi_check.isna().sum())
print("Duplicate dates:", cpi_check["date"].duplicated().sum())

start_date = pd.Timestamp("2015-01-01")
end_date = pd.Timestamp("2025-12-01")
if cpi_check["date"].min() > start_date:
    raise ValueError("CPI data does not include January 2015.")

if cpi_check["date"].max() < end_date:
    raise ValueError("CPI data does not include December 2025.")

print("CPI data contains the required period.")
display(cpi_check.head())


Total observations: 132
Earliest date: 2015-01-01 00:00:00
Latest date: 2025-12-01 00:00:00
Missing values:
observation_date    0
CPIAUCSL            1
date                0
dtype: int64
Duplicate dates: 0
CPI data contains the required period.


,observation_date,CPIAUCSL,date
0,2015-01-01,234.747,2015-01-01
1,2015-02-01,235.342,2015-02-01
2,2015-03-01,235.976,2015-03-01
3,2015-04-01,236.222,2015-04-01
4,2015-05-01,237.001,2015-05-01


### Acquisition Error Handling

During my first few attempts, the FRED CSV endpoint timed out with both 30 second and 60 second limits. There is code included that raises an error for this event.

To avoid repeated requests, the notebook uses a cached CPI file included in the uploads area of Colab. The validation function also checks whether the data is empty or missing required columns.

I also tested the validation function with an incorrect CSV to show that it detects incorrect data and notify the user that it failed.

In [30]:
invalid_cpi = b"observation_date,WRONG_COLUMN\n2025-01-01,100\n"
try:
    validate_cpi(invalid_cpi)
    raise AssertionError("Incorrect CPI data was incorrectly accepted.")
except ValueError as e:
    print("Failure identified:", e)

Failure identified: A column = missing.


## 2. Ethics and Legality

For this project, I used publicly available economic data from the Bureau of Labor Statistics and Federal Reserve Economic Data. Neither source required a login or payment, also, the datasets do not contain personally identifiable information.

In my A8 acquisition plan, I planned to use the BLS Public Data API and downloading CPI data as a CSV file from FRED. My P3 followed this plan, however, with a necessary change. After experiencing timeouts when requesting the FRED data, I used a cached CSV file to avoid unnecessary repeated requests.

The 4 legal/ethical categories in A8:

- Hacking (CFAA): The project does not access restricted systems nor require authentication.
- Contract (Terms of Service): I reviewed the source terms in A8 and identified the requirement to provide credit to BLS and FRED.
- Circumvention: The acquisition process does not bypass paywalls, access restrictions, or other technical protections.
- Privacy: The datasets do not have information about identifiable individuals.

#### Actions Taken:
The code utilizes an User-Agent, request timeouts, and caching. I saved the raw CPI data and documented the missing October 2025 CPI observation rather than changing it.

Overall, the acquisition approach remained mostly consistent with my A8 plan, and the project uses public data without the need to bypass any access restrictions.

In [40]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BLS_PATH = PROCESSED_DIR / "bls_hourly_earnings_cleaned.csv"
if not BLS_PATH.exists():
    raise FileNotFoundError(
        "LINDSEY UPLOAD FILE")
bls_clean = pd.read_csv(BLS_PATH, parse_dates=["date"])
assert len(bls_clean) == 1056, "Incorrect row count"
assert bls_clean.shape[1] == 8, "Incorrect column count"
assert bls_clean["date"].notna().all(), "Missing dates"

print("BLS dataset loaded.")
print("Shape:", bls_clean.shape)
print("File:", BLS_PATH)
display(bls_clean.head())

BLS dataset loaded.
Shape: (1056, 8)
File: data/processed/bls_hourly_earnings_cleaned.csv


,series_id,industry,year,period,period_name,hourly_earnings,month,date
0,CES2000000003,Construction,2024,M12,December,38.94,12,2024-12-01
1,CES2000000003,Construction,2024,M11,November,38.77,11,2024-11-01
2,CES2000000003,Construction,2024,M10,October,38.69,10,2024-10-01
3,CES2000000003,Construction,2024,M09,September,38.55,9,2024-09-01
4,CES2000000003,Construction,2024,M08,August,38.34,8,2024-08-01


In [41]:
BLS_PATH = PROCESSED_DIR / "bls_hourly_earnings_cleaned.csv"

bls_clean = pd.read_csv(BLS_PATH, parse_dates=["date"])
print("BLS dataset shape:", bls_clean.shape)
print("Columns:", bls_clean.columns.tolist())

display(bls_clean.head())

BLS dataset shape: (1056, 8)
Columns: ['series_id', 'industry', 'year', 'period', 'period_name', 'hourly_earnings', 'month', 'date']


,series_id,industry,year,period,period_name,hourly_earnings,month,date
0,CES2000000003,Construction,2024,M12,December,38.94,12,2024-12-01
1,CES2000000003,Construction,2024,M11,November,38.77,11,2024-11-01
2,CES2000000003,Construction,2024,M10,October,38.69,10,2024-10-01
3,CES2000000003,Construction,2024,M09,September,38.55,9,2024-09-01
4,CES2000000003,Construction,2024,M08,August,38.34,8,2024-08-01


In [42]:

print("BLS date range:")
print(bls_clean["date"].min(), "to", bls_clean["date"].max())

print("CPI date range:")
print(cpi_check["date"].min(), "to", cpi_check["date"].max())

print("BLS unique industries:", bls_clean["industry"].nunique())

print("BLS unique months:", bls_clean["date"].nunique())

print("CPI unique months:", cpi_check["date"].nunique())

print("Duplicate BLS industry-date combinations:", bls_clean.duplicated(subset=["industry", "date"]).sum())
print("Duplicate CPI dates:", cpi_check["date"].duplicated().sum())
missing_dates = (set(bls_clean["date"]) - set(cpi_check["date"]))
print("BLS dates missing from CPI:", len(missing_dates))
if missing_dates:
    print("Missing dates:", sorted(missing_dates))


BLS date range:
2015-01-01 00:00:00 to 2025-12-01 00:00:00
CPI date range:
2015-01-01 00:00:00 to 2025-12-01 00:00:00
BLS unique industries: 8
BLS unique months: 132
CPI unique months: 132
Duplicate BLS industry-date combinations: 0
Duplicate CPI dates: 0
BLS dates missing from CPI: 0


## 3. Join Design

| Join Keys | Answer |
|---|---|
| Join key | date |
| Grain of left table | One row per industry per month |
| Grain of right table | One row per month |
| Relationship | Many-to-one |
| Expected row count after join | 1,056 |
| Join type | Left join |
| What happens to the Unmatched rows | Kept with missing CPI values |


#### Why Many-To-One
A many-to-one relationship is most expected because the BLS dataset contains many industry observations for one month, but the CPI dataset only has one observation per month.

#### Join Type and Why
I selected a left join because the BLS wage observations are the main focus of my project. An inner join might delete BLS observations if a month was missing from the CPI dataset. A left join saves these observations and shows the missing CPI matches.

The merge will use validate="many_to_one" to ensure that the CPI dataset does not have duplicate monthly values. This helps prevent a fan out join, where a single BLS observation might incorrectly match more than one CPI records.

The expected result is 1,056 rows, as each CPI month is unique.

In [46]:
cpi_merge = cpi_check[["date", "CPIAUCSL"]].copy()
assert cpi_merge["date"].is_unique, ("CPI has duplicate dates!!!")
merged_df = pd.merge(bls_clean, cpi_merge, on="date", how="left", validate="many_to_one", indicator=True)
print("BLS rows before merge:", len(bls_clean))
print("Rows after merge:", len(merged_df))
print("Merge results:")
print(merged_df["_merge"].value_counts())
print("Merged dataset:")
display(merged_df.head())


BLS rows before merge: 1056
Rows after merge: 1056
Merge results:
_merge
both          1056
left_only        0
right_only       0
Name: count, dtype: int64
Merged dataset:


,series_id,industry,year,period,period_name,hourly_earnings,month,date,CPIAUCSL,_merge
0,CES2000000003,Construction,2024,M12,December,38.94,12,2024-12-01,317.604,both
1,CES2000000003,Construction,2024,M11,November,38.77,11,2024-11-01,316.528,both
2,CES2000000003,Construction,2024,M10,October,38.69,10,2024-10-01,315.631,both
3,CES2000000003,Construction,2024,M09,September,38.55,9,2024-09-01,314.732,both
4,CES2000000003,Construction,2024,M08,August,38.34,8,2024-08-01,314.062,both


## 4. Verification Suite

After combining the BLS hourly earnings data with the FRED Consumer Price Index data, I enforced a verification suite to check whether the merge performed correctly and produced the expected results.

The checks consist of validating row count preservation, unique industry date combinations, unique CPI key, missing matches, and valid CPI values.

These assertions are important because a merge can cause incorrect results and not show an error. For instance, duplicate dates in the CPI dataset could increase BLS observations, but missing CPI dates could leave wage values without connected inflation data.

In [47]:
# This ensures that the merge does not add or delete any rows.
assert len(merged_df) == len(bls_clean), ("Row count changed during the merge.")
# This protects duplicate industry month records.
assert not merged_df.duplicated(subset=["industry", "date"]).any(), ("Duplicate industry-date combinations found.")
# Makes sure that CPI dates are unique / no duplicate matches.
assert cpi_merge["date"].is_unique, ("CPI dates are not unique.")
# Ensures there are not unmatched dates.
assert (merged_df["_merge"] == "both").all(), ("Some BLS dates did not match CPI dates.")
# Checks that all CPI values are positive.
assert (merged_df["CPIAUCSL"].dropna() > 0).all(), ("Invalid CPI values found.")

# Makes sure that missing CPI values are in the same months as the source data.
source_missing_dates = set(cpi_merge.loc[cpi_merge["CPIAUCSL"].isna(), "date"])
merged_missing_dates = set(merged_df.loc[merged_df["CPIAUCSL"].isna(), "date"])
assert merged_missing_dates == source_missing_dates, ("Merged CPI missing dates differ from source.")

# Confirms that the 8 missing CPI values found are present.
assert merged_df["CPIAUCSL"].isna().sum() == 8, ("Unexpected number of missing CPI values.")

print("6/6 Pass")
print("Rows preserved:", len(merged_df))
print("Unique industries:", merged_df["industry"].nunique())
print("Unique months:", merged_df["date"].nunique())
print("Missing CPI values:", merged_df["CPIAUCSL"].isna().sum())
print("Missing CPI month(s):", sorted(merged_missing_dates))


6/6 Pass
Rows preserved: 1056
Unique industries: 8
Unique months: 132
Missing CPI values: 8
Missing CPI month(s): [Timestamp('2025-10-01 00:00:00')]


### 4. Verification Results and Missing Data

The verification suite confirmed that the integration kept all 1,056 BLS observations without creating duplicate industry-month combinations. All BLS dates successfully matched dates in the CPI dataset.

However, the first verification showed 8 missing CPI values in the merged dataset. After I spent time reviewing my code and data, it showed that these missing values were from a missing CPI observation for October 2025 in the original FRED CSV file.

Because the BLS dataset contains eight industries for each month, the  missing CPI observation was presented in eight merged rows.

I chose to keep these missing values rather than replace them with estimated CPI values or delete the related wage observations to maintain the original source data and ensure that estimated values did not affect my analysis.

The revised verification suite verfies that the missing values are consistent with the source dataset, also checks row preservation, unique keys, successful date matching, and valid nonmissing CPI values.

For any calculations requiring CPI, the eight affected observations will need to be deleted from those calculations or handled with a different method.

## 5. Surprising Finding

One unexpected finding during the integration process includes that having matching dates between two datasets does not ensure that the corresponding data values are fully complete.

Although all 1,056 BLS observations matched a date in the CPI dataset, eight records still contained missing CPI values. This happened since the October 2025 CPI record was included in the source file but did not have a reported CPI value. This finding showed why simply just checking only row counts and successful join matches is not the best way to verify that the merge was successful, or find an error. Additional verification of actual data values was quite necessary to ensure that an integrated dataset is ready for analysis. It also presented the importance of preserving missing data and documenting its impact instead of filling in missing values.

In [49]:
VERIFIED_PATH = (PROCESSED_DIR / "bls_cpi_merged_verified.csv")

merged_df.to_csv(VERIFIED_PATH, index=False)

print("Verified dataset saved successfully!")
print("File:", VERIFIED_PATH)
print("Shape:", merged_df.shape)
print("Missing CPI values:", merged_df["CPIAUCSL"].isna().sum())


Verified dataset saved successfully!
File: data/processed/bls_cpi_merged_verified.csv
Shape: (1056, 10)
Missing CPI values: 8


## 6. Self-Scored Rubric

| Criterion | Wt | The short version |
|---|---|---|
| Acquisition code | 2 | Retrieves reliably, rate-limited, cached, and fails loudly, with one demonstrated catch |
| Ethics & legality in practice | 2 | Matches the A8 plan; drift documented |
| Join design | 2 | Keys and cardinality stated **before** the merge; alternatives ruled out |
| **Verification suite** | **3** | three or more assertions |
| Reproducibility & readability | 0.5 | Runs without errors, readable by a peer, README updated |